#Feature Products

##Data Definitions

In [0]:
CATALOG_NAME = "ml_training_dev"
SOURCE_SCHEMA_NAME = "gold"
TARGET_SCHEMA_NAME = "feature"
SOURCE_TABLE_NAME = "customers_orders_category"
TARGET_TABLE_NAME = "products"

##Create Catalog and schema

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS {}.{}".format(CATALOG_NAME, TARGET_SCHEMA_NAME))

##Import Libraries

In [0]:
from pyspark.sql.functions import current_date, count,sum, avg, round, max, to_date,col, min, countDistinct, row_number,first
from pyspark.sql.window import Window

##Reading Source Tables

In [0]:
df_feature_table = spark.table(f"{CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.{SOURCE_TABLE_NAME}").drop("loadDate")

##Transformations

In [0]:
df_final = (
    df_feature_table
    .groupBy("product_id")
    .agg(
        first("product_category_name", ignorenulls=True)
            .alias("product_category_name"),

        round(avg("price"), 2)
            .alias("avg_price"),

        round(avg("freight_value"), 2)
            .alias("avg_freight_value"),

        first("product_weight_g", ignorenulls=True)
            .alias("product_weight_g"),

        first("product_length_cm", ignorenulls=True)
            .alias("product_length_cm"),

        first("product_height_cm", ignorenulls=True)
            .alias("product_height_cm"),

        first("product_width_cm", ignorenulls=True)
            .alias("product_width_cm"),

        countDistinct("order_id")
            .alias("total_orders"),

        countDistinct("customer_unique_id")
            .alias("unique_customers"),

        round(sum("price"), 2)
            .alias("total_revenue"),

        round(avg("review_score"), 2)
            .alias("avg_rating"),

        min("order_purchase_timestamp")
            .alias("first_sale_date"),

        max("order_purchase_timestamp")
            .alias("last_sale_date")
    )
)

In [0]:
# df_final.display()

##Writting table

In [0]:
df_final.write.mode("overwrite").saveAsTable(f"{CATALOG_NAME}.{TARGET_SCHEMA_NAME}.{TARGET_TABLE_NAME}")
print("table successfully created!")

In [0]:
# Cliente A ── Produto 1
#          ├─ Produto 2
#          └─ Produto 3

# Cliente B ── Produto 1
#          ├─ Produto 2
#          └─ Produto 4

# A → recomenda Produto 4

In [0]:
# Cliente gosta de:

# categoria = beleza
# preço = 50-100
# frete = baixo

In [0]:
# customer_id
# product_id
# customer_features
# product_features
# interaction_features
#         │
#         ▼
# Logistic Regression
#         │
#         ▼
# P(buy product | customer)



# Cliente A + Produto X → 0.82
# Cliente A + Produto Y → 0.63
# Cliente A + Produto Z → 0.12



# X
# Y